In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io
import glob
from datetime import datetime
from pysteps import io, nowcasts, verification
from pysteps.motion.lucaskanade import dense_lucaskanade
from pysteps.utils import conversion, dimension, transformation
from pysteps.postprocessing import ensemblestats
from sklearn.metrics import mean_squared_error
from math import sqrt
from matplotlib.gridspec import GridSpec

Pysteps configuration file found at: c:\Users\LENOVO\anaconda3\envs\vyrginia\Lib\site-packages\pysteps\pystepsrc



In [2]:
#Configuration
DATE_STR = "20221109"  
NOWCAST_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/INPUT_NOWCAST/"
VERIFICATION_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/"
OUTPUT_DIR = "D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/verifications/"
N_LEADTIMES = 30  
N_ENS_MEMBERS = 10
SEED = 24
THRESHOLDS = [5.0, 10.0, 20.0] 
ACCUTIME = 2  
SELECTED_LEADTIMES = [10, 20, 30, 40, 50, 60]  

In [3]:
def load_santanu_data(input_dir):
    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Folder {input_dir} does not exist.")
    
    R = []
    times = []
    
    for filename in sorted(os.listdir(input_dir)):
        if filename.endswith('.mat'):
            file_path = os.path.join(input_dir, filename)
            mat_data = scipy.io.loadmat(file_path)
            
            if 'ZI' not in mat_data:
                continue
            
            reflectivity = mat_data['ZI'] / 10000  # Normalize
            reflectivity[reflectivity == -1] = np.nan
            R.append(reflectivity)
            
            if 'f' in mat_data and 't' in mat_data:
                time_str = mat_data['f'][0][0]
                times.append(int(time_str))
    
    if not R:
        raise ValueError("No valid reflectivity data found.")
    
    metadata = {
        "institution": "Santanu",
        "accutime": ACCUTIME,
        "unit": "mm/h",
        "transform": "dB",
        "zerovalue": 0,
        "threshold": 0.1,
        'x1': 107, 'x2': 108,
        'y1': -7.2, 'y2': -6.4,
        'lat': np.linspace(-7.2, -6.4, R[0].shape[0]),
        'lon': np.linspace(107, 108, R[0].shape[1]),
        'projection': "+proj=merc +lon_0=0 +datum=WGS84 +units=m +no_defs",
        'yorigin': 'upper',
        "cartesian_map": True,
        'zr_a': 316.0, 'zr_b': 1.5,
    }
    
    return np.array(R), np.array(times), metadata

In [4]:
def perform_nowcast(R, metadata):
    #Transform data
    R, metadata = transformation.dB_transform(R, metadata, threshold=0.1, zerovalue=-15.0)
    R[~np.isfinite(R)] = -15.0
    
    #Estimate motion field
    V = dense_lucaskanade(R)
    
    #Perform nowcast
    nowcast_method = nowcasts.get_method("steps")
    R_f_steps = nowcast_method(
        R[-3:, :, :],
        V,
        N_LEADTIMES,
        N_ENS_MEMBERS,
        n_cascade_levels=6,
        precip_thr=10,
        kmperpixel=0.12,
        timestep=metadata["accutime"],
        noise_method="nonparametric",
        vel_pert_method="bps",
        mask_method="incremental",
        seed=SEED,
    )
    
    #Back-transform to rain rates
    return transformation.dB_transform(R_f_steps, threshold=10.0, inverse=True)[0]


In [5]:
def load_verification_data(input_dir, n_leadtimes):
    fns = glob.glob(os.path.join(input_dir, "*.mat"))
    fns.sort()
    
    if len(fns) < n_leadtimes:
        print(f"Warning: Only {len(fns)} observation files available (requested {n_leadtimes})")
    
    R_o = []
    for fn in fns[-n_leadtimes:]:
        mat_data = scipy.io.loadmat(fn)
        if 'ZI' in mat_data:
            reflectivity = mat_data['ZI'] / 10000
            reflectivity[reflectivity == -1] = np.nan
            R_o.append(reflectivity)
    
    return np.array(R_o)

In [6]:
def calculate_metrics(R_f_steps, R_o, metadata):
    n_leadtimes = min(R_f_steps.shape[1], R_o.shape[0])
    lead_times = [(lt+1)*metadata['accutime'] for lt in range(n_leadtimes)]
    
    if n_leadtimes < R_f_steps.shape[1]:
        print(f"Warning: Only verifying {n_leadtimes} lead times (of {R_f_steps.shape[1]} predicted)")
    
    metrics = {
        'lead_times': lead_times,
        'rmse': [],
        'correlation': [],
        'fss': {thr: [] for thr in THRESHOLDS},
        'roc_auc': {thr: [] for thr in THRESHOLDS},
        'roc_curves': {thr: {} for thr in THRESHOLDS},
        'reldiags': {thr: {} for thr in THRESHOLDS},
    }
    
    for lt in range(n_leadtimes):
        current_lead_time = (lt+1)*metadata['accutime']
        R_f_mean = np.mean(R_f_steps[:, lt, :, :], axis=0)
        R_o_lt = R_o[lt, :, :]
        
        valid_mask = np.isfinite(R_f_mean) & np.isfinite(R_o_lt)
        if np.sum(valid_mask) > 0:
            rmse = sqrt(mean_squared_error(R_f_mean[valid_mask], R_o_lt[valid_mask]))
            corr = np.corrcoef(R_f_mean[valid_mask].flatten(), R_o_lt[valid_mask].flatten())[0, 1]
        else:
            rmse = np.nan
            corr = np.nan
        
        metrics['rmse'].append(rmse)
        metrics['correlation'].append(corr)
        
        for thr in THRESHOLDS:
            #Calculate FSS
            metrics['fss'][thr].append(verification.fss(R_f_mean, R_o_lt, thr, scale=10))
            
            #Calculate ROC curves and AUC
            P_f = ensemblestats.excprob(R_f_steps[:, lt, :, :], thr)
            
            #Store ROC curve for selected lead times
            if current_lead_time in SELECTED_LEADTIMES:
                roc = verification.ROC_curve_init(thr, n_prob_thrs=20)
                verification.ROC_curve_accum(roc, P_f, R_o_lt)
                metrics['roc_curves'][thr][current_lead_time] = roc
                
                #Store reliability diagram
                reldiag = verification.reldiag_init(thr)
                verification.reldiag_accum(reldiag, P_f, R_o_lt)
                metrics['reldiags'][thr][current_lead_time] = reldiag
            
            #Compute ROC AUC for all lead times
            roc = verification.ROC_curve_init(thr)
            verification.ROC_curve_accum(roc, P_f, R_o_lt)
            roc_result = verification.ROC_curve_compute(roc)
            metrics['roc_auc'][thr].append(roc_result[2] if len(roc_result) >= 3 else np.nan)
    
    return metrics

In [7]:
def create_rmse_plot(metrics, output_dir, date_str):
    plt.figure(figsize=(12, 6))
    
    #Convert date string to formatted date
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    #Create plot
    ax = plt.gca()
    ax.plot(metrics['lead_times'], metrics['rmse'], 
           color='#1f77b4', marker='o', linestyle='-', linewidth=2, markersize=8)
    
    #Highlight selected lead times
    for lt in SELECTED_LEADTIMES:
        if lt in metrics['lead_times']:
            idx = metrics['lead_times'].index(lt)
            ax.plot(lt, metrics['rmse'][idx], 'ro', markersize=8)
    
    #Add annotations
    max_idx = np.nanargmax(metrics['rmse'])
    min_idx = np.nanargmin(metrics['rmse'])
    ax.annotate(f'Max: {metrics["rmse"][max_idx]:.2f} mm/h', 
               xy=(metrics['lead_times'][max_idx], metrics['rmse'][max_idx]),
               xytext=(10,10), textcoords='offset points',
               arrowprops=dict(arrowstyle='->', color='red'))
    ax.annotate(f'Min: {metrics["rmse"][min_idx]:.2f} mm/h', 
               xy=(metrics['lead_times'][min_idx], metrics['rmse'][min_idx]),
               xytext=(10,-20), textcoords='offset points',
               arrowprops=dict(arrowstyle='->', color='green'))
    
    #Formatting
    ax.set_xlabel('Lead Time (minutes)', fontsize=12, fontweight='bold')
    ax.set_ylabel('RMSE (mm/h)', fontsize=12, fontweight='bold')
    ax.set_title(f'RMSE vs Lead Time\n{formatted_date}', fontsize=14, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xticks(metrics['lead_times'])
    ax.set_xticklabels([f"{int(t)}" for t in metrics['lead_times']], rotation=45)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'RMSE_enhanced.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [8]:
def create_roc_plots(metrics, output_dir, date_str):
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    #Create one plot per threshold
    for thr in THRESHOLDS:
        plt.figure(figsize=(10, 8))
        
        #Create grid for subplots
        gs = GridSpec(2, 3)
        ax1 = plt.subplot(gs[0, 0])
        ax2 = plt.subplot(gs[0, 1])
        ax3 = plt.subplot(gs[0, 2])
        ax4 = plt.subplot(gs[1, 0])
        ax5 = plt.subplot(gs[1, 1])
        ax6 = plt.subplot(gs[1, 2])
        
        axes = [ax1, ax2, ax3, ax4, ax5, ax6]
        
        for ax, lt in zip(axes, SELECTED_LEADTIMES):
            if lt in metrics['roc_curves'][thr]:
                roc = metrics['roc_curves'][thr][lt]
                try:
                    # Plot the ROC curve
                    verification.plot_ROC(roc, ax, opt_prob_thr=True)
                    ax.set_title(f"+{lt} min", fontsize=10)
                    ax.grid(True, linestyle='--', alpha=0.6)
                    
                    # Compute ROC metrics
                    roc_result = verification.ROC_curve_compute(roc)
                    
                    # Add AUC score to plot
                    if len(roc_result) >= 3:  # POD, POFD, AUC
                        auc_score = float(roc_result[2])
                        ax.text(0.6, 0.1, f'AUC: {auc_score:.3f}', 
                                transform=ax.transAxes, 
                                bbox=dict(facecolor='white', alpha=0.8))
                    else:
                        ax.text(0.6, 0.1, 'AUC: N/A', 
                                transform=ax.transAxes, 
                                bbox=dict(facecolor='white', alpha=0.8))
                except Exception as e:
                    print(f"Error processing ROC curve for threshold {thr}mm at {lt}min: {str(e)}")
                    ax.text(0.5, 0.5, 'ROC Error', 
                            ha='center', va='center',
                            transform=ax.transAxes, 
                            bbox=dict(facecolor='white', alpha=0.8))
        
        plt.suptitle(f"ROC Curves for {thr} mm/h Threshold\n{formatted_date}", 
                    fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'ROC_curves_{thr}mm.png'), 
                    dpi=300, bbox_inches='tight')
        plt.close()

In [9]:
def create_reliability_diagrams(metrics, output_dir, date_str):
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    #Create one plot per threshold
    for thr in THRESHOLDS:
        plt.figure(figsize=(10, 8))
        
        #Create grid for subplots
        gs = GridSpec(2, 3)
        ax1 = plt.subplot(gs[0, 0])
        ax2 = plt.subplot(gs[0, 1])
        ax3 = plt.subplot(gs[0, 2])
        ax4 = plt.subplot(gs[1, 0])
        ax5 = plt.subplot(gs[1, 1])
        ax6 = plt.subplot(gs[1, 2])
        
        axes = [ax1, ax2, ax3, ax4, ax5, ax6]
        
        for ax, lt in zip(axes, SELECTED_LEADTIMES):
            if lt in metrics['reldiags'][thr]:
                reldiag = metrics['reldiags'][thr][lt]
                try:
                    verification.plot_reldiag(reldiag, ax)
                    ax.set_title(f"+{lt} min", fontsize=10)
                    ax.grid(True, linestyle='--', alpha=0.6)
                    
                    #Compute reliability diagram statistics
                    reldiag_result = verification.reldiag_compute(reldiag)
                    
                    #Add Brier score to plot
                    if len(reldiag_result) >= 5:  # Typical return has Brier score last
                        bs = float(reldiag_result[-1])
                        ax.text(0.6, 0.1, f'BS: {bs:.3f}', 
                                transform=ax.transAxes, 
                                bbox=dict(facecolor='white', alpha=0.8))
                    else:
                        ax.text(0.6, 0.1, 'BS: N/A', 
                                transform=ax.transAxes, 
                                bbox=dict(facecolor='white', alpha=0.8))
                except Exception as e:
                    print(f"Error processing reliability diagram for threshold {thr}mm at {lt}min: {str(e)}")
                    ax.text(0.5, 0.5, 'Reliability Error', 
                            ha='center', va='center',
                            transform=ax.transAxes, 
                            bbox=dict(facecolor='white', alpha=0.8))
        
        plt.suptitle(f"Reliability Diagrams for {thr} mm/h Threshold\n{formatted_date}", 
                    fontsize=12, fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'Reliability_diagrams_{thr}mm.png'), 
                    dpi=300, bbox_inches='tight')
        plt.close()

def save_results(metrics, output_dir, date_str):
    """Save all results and create visualizations"""
    os.makedirs(output_dir, exist_ok=True)
    np.save(os.path.join(output_dir, "verification_metrics.npy"), metrics)
    
    #Create enhanced plots
    create_rmse_plot(metrics, output_dir, date_str)
    create_roc_plots(metrics, output_dir, date_str)
    create_reliability_diagrams(metrics, output_dir, date_str)

def main():
    """Main workflow"""
    date = datetime.strptime(DATE_STR, "%Y%m%d")
    
    print("Loading nowcast input data...")
    R, _, metadata = load_santanu_data(NOWCAST_INPUT_DIR)
    
    print("Performing nowcast...")
    R_f_steps = perform_nowcast(R, metadata)
    
    print("Loading verification data...")
    R_o = load_verification_data(VERIFICATION_INPUT_DIR, N_LEADTIMES)
    R_o, _ = conversion.to_rainrate(R_o, metadata)
    
    print("Calculating metrics...")
    metrics = calculate_metrics(R_f_steps, R_o, metadata)
    
    print("Saving results...")
    save_results(metrics, OUTPUT_DIR, DATE_STR)
    
    print(f"Process completed. Results saved to {OUTPUT_DIR}")

if __name__ == "__main__":
    main()

Loading nowcast input data...
Performing nowcast...
Computing STEPS nowcast
-----------------------

Inputs
------
input dimensions: 734x734
km/pixel:         0.12
time step:        2 minutes

Methods
-------
extrapolation:          semilagrangian
bandpass filter:        gaussian
decomposition:          fft
noise generator:        nonparametric
noise adjustment:       no
velocity perturbator:   bps
conditional statistics: no
precip. mask method:    incremental
probability matching:   cdf
FFT method:             numpy
domain:                 spatial

Parameters
----------
number of time steps:     30
ensemble size:            10
parallel threads:         1
number of cascade levels: 6
order of the AR(p) model: 2
velocity perturbations, parallel:      10.88,0.23,-7.68
velocity perturbations, perpendicular: 5.76,0.31,-2.72
precip. intensity threshold: 10
************************************************
* Correlation coefficients for cascade levels: *
***************************************

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9712\1194380680.py:50: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9712\1194380680.py:50: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_9712\1194380680.py:50: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Process completed. Results saved to D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/verifications/


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io
import glob
from datetime import datetime
import pandas as pd
from pysteps import io, nowcasts, verification
from pysteps.motion.lucaskanade import dense_lucaskanade
from pysteps.utils import conversion, dimension, transformation
from pysteps.postprocessing import ensemblestats
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.gridspec as gridspec
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configuration
DATE_STR = "20221109"  
NOWCAST_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/INPUT_NOWCAST/"
VERIFICATION_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/"
OUTPUT_DIR = "D:/College/MENEA/FIXED_DATA/OUTPUT/20210616/verifications/"
N_LEADTIMES = 30  
N_ENS_MEMBERS = 10
SEED = 24
THRESHOLDS = [5.0, 10.0, 20.0]  # Rainfall intensity thresholds (mm/h)
ACCUTIME = 2  # Time between frames in minutes
SELECTED_LEADTIMES = [10, 20, 30, 40, 50, 60]  

# Enhanced Style configuration
plt.style.use('seaborn-v0_8-darkgrid')
COLORS = plt.cm.viridis(np.linspace(0, 1, len(THRESHOLDS)))
MARKERS = ['o', 's', 'D', '^', 'v', '<', '>', 'p', '*', 'h', 'H', '+', 'x', 'X', 'd', '|', '_']
LINE_STYLES = ['-', '--', '-.', ':']
FONT_SIZE = 20  # Increased base font size
plt.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.titlesize': FONT_SIZE + 2,
    'axes.labelsize': FONT_SIZE + 1,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'figure.titlesize': FONT_SIZE + 4,
    'figure.facecolor': 'white',
    'figure.edgecolor': 'white'
})

def load_santanu_data(input_dir):
    """Load radar reflectivity data from .mat files"""
    print(f"\nLoading data from: {input_dir}")
    
    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Folder {input_dir} does not exist.")
    
    R = []
    times = []
    files_loaded = 0
    
    for filename in sorted(os.listdir(input_dir)):
        if filename.endswith('.mat'):
            try:
                file_path = os.path.join(input_dir, filename)
                mat_data = scipy.io.loadmat(file_path)
                
                if 'ZI' not in mat_data:
                    continue
                
                reflectivity = mat_data['ZI'] / 10000  # Normalize
                reflectivity[reflectivity == -1] = np.nan
                
                if np.all(np.isnan(reflectivity)):
                    print(f"Warning: All NaN values in {filename}")
                    continue
                
                R.append(reflectivity)
                files_loaded += 1
                
            except Exception as e:
                print(f"Error loading {filename}: {str(e)}")
                continue
    
    if not R:
        raise ValueError("No valid reflectivity data found.")
    
    print(f"Successfully loaded {files_loaded} files")
    
    metadata = {
        "institution": "Santanu",
        "accutime": ACCUTIME,
        "unit": "mm/h",
        "transform": "dB",
        "zerovalue": 0,
        "threshold": 0.1,
        'x1': 107, 'x2': 108,
        'y1': -7.2, 'y2': -6.4,
        'lat': np.linspace(-7.2, -6.4, R[0].shape[0]),
        'lon': np.linspace(107, 108, R[0].shape[1]),
        'projection': "+proj=merc +lon_0=0 +datum=WGS84 +units=m +no_defs",
        'yorigin': 'upper',
        "cartesian_map": True,
        'zr_a': 316.0, 'zr_b': 1.5,
    }
    
    return np.array(R), np.array(times), metadata

def perform_nowcast(R, metadata):
    """Perform STEPS nowcasting"""
    print("\nStarting nowcast process...")
    
    try:
        # Validate input data
        if np.all(np.isnan(R[-3:])):
            raise ValueError("Input data contains only NaN values")
            
        print("- Original data stats:")
        print(f"  Shape: {R.shape}")
        print(f"  Min: {np.nanmin(R):.2f}, Max: {np.nanmax(R):.2f}")
        print(f"  NaN count: {np.isnan(R).sum()} ({np.isnan(R).mean()*100:.1f}%)")

        # Transform data
        print("- Applying dB transform...")
        R, metadata = transformation.dB_transform(
            R, 
            metadata, 
            threshold=0.1, 
            zerovalue=-15.0
        )
        
        print("- Post-transform stats:")
        print(f"  Min: {np.nanmin(R):.2f}, Max: {np.nanmax(R):.2f}")
        print(f"  NaN count: {np.isnan(R).sum()} ({np.isnan(R).mean()*100:.1f}%)")
        
        if np.all(np.isnan(R[-3:])):
            raise ValueError("All NaN values after dB transform")

        # Replace remaining NaN values
        R[~np.isfinite(R)] = -15.0
        
        # Estimate motion field
        print("- Calculating motion field...")
        V = dense_lucaskanade(R[-3:])  # Only use last 3 frames for motion estimation
        
        # Validate motion field
        if np.all(np.isnan(V)):
            raise ValueError("Motion field calculation failed (all NaN values)")

        # Perform nowcast
        print(f"- Running STEPS nowcast for {N_LEADTIMES} lead times...")
        nowcast_method = nowcasts.get_method("steps")
        R_f_steps = nowcast_method(
            R[-3:, :, :],  # Use last 3 frames
            V,
            N_LEADTIMES,
            N_ENS_MEMBERS,
            n_cascade_levels=6,
            precip_thr=0.1,
            kmperpixel=0.12,
            timestep=metadata["accutime"],
            noise_method="nonparametric",
            vel_pert_method="bps",
            mask_method="incremental",
            seed=SEED,
        )
        
        # Back-transform to rain rates
        print("- Converting back to rain rates...")
        R_f_steps, _ = transformation.dB_transform(
            R_f_steps, 
            threshold=0.1,
            inverse=True
        )
        
        return R_f_steps
    
    except Exception as e:
        print(f"Error in nowcast process: {str(e)}")
        print("Debug info:")
        if 'R' in locals():
            print(f"R shape: {R.shape if hasattr(R, 'shape') else 'N/A'}")
            print(f"R valid values: {np.sum(np.isfinite(R)) if hasattr(R, 'shape') else 'N/A'}")
        if 'V' in locals():
            print(f"V shape: {V.shape if hasattr(V, 'shape') else 'N/A'}")
        raise

def load_verification_data(input_dir, n_leadtimes):
    """Load verification data"""
    print(f"\nLoading verification data from: {input_dir}")
    
    fns = glob.glob(os.path.join(input_dir, "*.mat"))
    fns.sort()
    
    if not fns:
        raise FileNotFoundError(f"No .mat files found in {input_dir}")
    
    if len(fns) < n_leadtimes:
        print(f"Warning: Only {len(fns)} observation files available (requested {n_leadtimes})")
    
    R_o = []
    valid_files = 0
    
    for fn in fns[-n_leadtimes:]:
        try:
            mat_data = scipy.io.loadmat(fn)
            if 'ZI' in mat_data:
                reflectivity = mat_data['ZI'] / 10000
                reflectivity[reflectivity == -1] = np.nan
                
                if np.any(~np.isnan(reflectivity)):
                    R_o.append(reflectivity)
                    valid_files += 1
                else:
                    print(f"Warning: All NaN values in {os.path.basename(fn)}")
        except Exception as e:
            print(f"Error loading {os.path.basename(fn)}: {str(e)}")
    
    print(f"Loaded {valid_files} valid verification files")
    return np.array(R_o)

def calculate_metrics(R_f_steps, R_o, metadata):
    """Calculate verification metrics"""
    print("\nCalculating verification metrics...")
    
    n_leadtimes = min(R_f_steps.shape[1], R_o.shape[0])
    lead_times = [(lt+1)*metadata['accutime'] for lt in range(n_leadtimes)]
    
    metrics = {
        'lead_times': lead_times,
        'rmse': [],
        'correlation': [],
        'fss': {thr: [] for thr in THRESHOLDS},
        'bias': {thr: [] for thr in THRESHOLDS},
        'pod': {thr: [] for thr in THRESHOLDS},
        'far': {thr: [] for thr in THRESHOLDS},
        'csi': {thr: [] for thr in THRESHOLDS},
        'data_availability': {thr: [] for thr in THRESHOLDS},
    }

    for lt in range(n_leadtimes):
        current_lead_time = lead_times[lt]
        print(f"\nProcessing lead time: +{current_lead_time} minutes")
        
        R_f_mean = np.mean(R_f_steps[:, lt, :, :], axis=0)
        R_o_lt = R_o[lt, :, :]
        
        # Basic metrics
        valid_mask = np.isfinite(R_f_mean) & np.isfinite(R_o_lt)
        valid_count = np.sum(valid_mask)
        
        if valid_count > 0:
            metrics['rmse'].append(sqrt(mean_squared_error(R_f_mean[valid_mask], R_o_lt[valid_mask])))
            metrics['correlation'].append(np.corrcoef(R_f_mean[valid_mask].flatten(), R_o_lt[valid_mask].flatten())[0, 1])
        else:
            print(f"Warning: No valid pixels for basic metrics at +{current_lead_time}min")
            metrics['rmse'].append(np.nan)
            metrics['correlation'].append(np.nan)
        
        # Threshold-based metrics
        for thr in THRESHOLDS:
            obs_above = np.nansum(R_o_lt >= thr)
            fcst_above = np.nansum(R_f_mean >= thr)
            metrics['data_availability'][thr].append(obs_above)
            
            print(f"  Threshold {thr}mm/h - Obs above: {obs_above}, Fcst above: {fcst_above}")
            
            # FSS
            try:
                fss = verification.fss(R_f_mean, R_o_lt, thr, scale=10)
                metrics['fss'][thr].append(fss)
            except Exception as e:
                print(f"    FSS calculation failed: {str(e)}")
                metrics['fss'][thr].append(np.nan)
            
            # Binary contingency metrics
            obs_binary = (R_o_lt >= thr).astype(float)
            fcst_binary = (R_f_mean >= thr).astype(float)
            
            valid = np.isfinite(obs_binary) & np.isfinite(fcst_binary)
            obs_binary = obs_binary[valid]
            fcst_binary = fcst_binary[valid]
            
            if len(obs_binary) > 0 and np.any(obs_binary > 0):
                hits = np.sum((fcst_binary == 1) & (obs_binary == 1))
                false_alarms = np.sum((fcst_binary == 1) & (obs_binary == 0))
                misses = np.sum((fcst_binary == 0) & (obs_binary == 1))
                
                denom = hits + misses
                metrics['pod'][thr].append(hits / denom if denom > 0 else np.nan)
                
                denom = hits + false_alarms
                metrics['far'][thr].append(false_alarms / denom if denom > 0 else np.nan)
                metrics['csi'][thr].append(hits / (hits + false_alarms + misses) if (hits + false_alarms + misses) > 0 else np.nan)
                metrics['bias'][thr].append((hits + false_alarms) / (hits + misses) if (hits + misses) > 0 else np.nan)
            else:
                print(f"    Insufficient data for contingency metrics at {thr}mm/h")
                metrics['pod'][thr].append(np.nan)
                metrics['far'][thr].append(np.nan)
                metrics['csi'][thr].append(np.nan)
                metrics['bias'][thr].append(np.nan)
    
    return metrics

def plot_rmse(metrics, output_dir, date_str):
    """Plot RMSE with enhanced styling and larger text"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(12, 8))  # Increased figure size
    
    if any(not np.isnan(x) for x in metrics['rmse']):
        ax.plot(metrics['lead_times'], metrics['rmse'], 
                color='#1f77b4', marker='o', linestyle='-', 
                linewidth=3, markersize=12, label='RMSE')  # Increased line width and marker size
        
        # Highlight selected lead times
        for lt in SELECTED_LEADTIMES:
            if lt in metrics['lead_times']:
                idx = metrics['lead_times'].index(lt)
                if not np.isnan(metrics['rmse'][idx]):
                    ax.plot(lt, metrics['rmse'][idx], 'ro', markersize=14, 
                            markeredgecolor='k', linewidth=2,
                            label=f'Selected +{lt}min' if lt == SELECTED_LEADTIMES[0] else "")
        
        ax.set_ylim(0, np.nanmax(metrics['rmse']) * 1.1)
        ax.legend(fontsize=FONT_SIZE)
    else:
        ax.text(0.5, 0.5, 'No valid RMSE data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('RMSE (mm/h)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_title(f'RMSE vs Lead Time\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xticks(SELECTED_LEADTIMES)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'RMSE_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_correlation(metrics, output_dir, date_str):
    """Plot Correlation with enhanced styling"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(12, 8))  # Increased figure size
    
    if any(not np.isnan(x) for x in metrics['correlation']):
        ax.plot(metrics['lead_times'], metrics['correlation'], 
                color='#ff7f0e', marker='s', linestyle='--', 
                linewidth=3, markersize=12, label='Correlation')
        
        # Highlight selected lead times
        for lt in SELECTED_LEADTIMES:
            if lt in metrics['lead_times']:
                idx = metrics['lead_times'].index(lt)
                if not np.isnan(metrics['correlation'][idx]):
                    ax.plot(lt, metrics['correlation'][idx], 'ro', markersize=14, 
                            markeredgecolor='k', linewidth=2, label=f'Selected +{lt}min' if lt == SELECTED_LEADTIMES[0] else "")
        
        ax.set_ylim(-0.1, 1.1)
        ax.legend(fontsize=FONT_SIZE)
        ax.axhline(0.5, color='gray', linestyle=':', linewidth=2, label='Reference')
    else:
        ax.text(0.5, 0.5, 'No valid Correlation data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('Correlation Coefficient', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_title(f'Correlation vs Lead Time\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.set_xticks(SELECTED_LEADTIMES)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'Correlation_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_fss(metrics, output_dir, date_str):
    """Plot FSS for all thresholds with enhanced styling"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    # Create separate plots for each threshold
    for i, thr in enumerate(THRESHOLDS):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if any(not np.isnan(x) for x in metrics['fss'][thr]):
            ax.plot(metrics['lead_times'], metrics['fss'][thr], 
                    color=COLORS[i], marker=MARKERS[i], linestyle=LINE_STYLES[i%len(LINE_STYLES)], 
                    linewidth=3, markersize=12, label=f'{thr} mm/h')
            
            ax.axhline(0.5, color='k', linestyle='--', linewidth=3, label='Skill threshold')
            ax.set_ylim(0, 1.05)
            ax.legend(fontsize=FONT_SIZE)
            ax.set_title(f'Fractions Skill Score (FSS) - {thr} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'No valid FSS data for {thr} mm/h', 
                    ha='center', va='center', fontsize=FONT_SIZE+2)
        
        ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.set_ylabel('FSS Score', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xticks(SELECTED_LEADTIMES)
        ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'FSS_plot_{int(thr)}mm.png'), dpi=300, bbox_inches='tight')
        plt.close()

def plot_pod(metrics, output_dir, date_str):
    """Plot Probability of Detection (POD) for all thresholds as separate plots"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    # Create separate plots for each threshold
    for i, thr in enumerate(THRESHOLDS):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if any(not np.isnan(x) for x in metrics['pod'][thr]):
            ax.plot(metrics['lead_times'], metrics['pod'][thr], 
                    color=COLORS[i], marker=MARKERS[i], linestyle=LINE_STYLES[i%len(LINE_STYLES)], 
                    linewidth=3, markersize=12, label=f'{thr} mm/h')
            
            ax.axhline(0.6, color='r', linestyle='--', linewidth=2, label='Threshold (0.6)')
            ax.set_ylim(0, 1.05)
            ax.legend(fontsize=FONT_SIZE)
            ax.set_title(f'Probability of Detection (POD) - {thr} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'No valid POD data for {thr} mm/h', 
                    ha='center', va='center', fontsize=FONT_SIZE+2)
        
        ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.set_ylabel('POD', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xticks(SELECTED_LEADTIMES)
        ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'POD_plot_{int(thr)}mm.png'), dpi=300, bbox_inches='tight')
        plt.close()

def plot_far(metrics, output_dir, date_str):
    """Plot False Alarm Ratio (FAR) for all thresholds as separate plots"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    # Create separate plots for each threshold
    for i, thr in enumerate(THRESHOLDS):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if any(not np.isnan(x) for x in metrics['far'][thr]):
            ax.plot(metrics['lead_times'], metrics['far'][thr], 
                    color=COLORS[i], marker=MARKERS[i], linestyle=LINE_STYLES[i%len(LINE_STYLES)], 
                    linewidth=3, markersize=12, label=f'{thr} mm/h')
            
            ax.axhline(0.4, color='b', linestyle='--', linewidth=2, label='Threshold (0.4)')
            ax.set_ylim(0, 1.05)
            ax.legend(fontsize=FONT_SIZE)
            ax.set_title(f'False Alarm Ratio (FAR) - {thr} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'No valid FAR data for {thr} mm/h', 
                    ha='center', va='center', fontsize=FONT_SIZE+2)
        
        ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.set_ylabel('FAR', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xticks(SELECTED_LEADTIMES)
        ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'FAR_plot_{int(thr)}mm.png'), dpi=300, bbox_inches='tight')
        plt.close()

def plot_csi(metrics, output_dir, date_str):
    """Plot Critical Success Index (CSI) for all thresholds as separate plots"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    # Create separate plots for each threshold
    for i, thr in enumerate(THRESHOLDS):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if any(not np.isnan(x) for x in metrics['csi'][thr]):
            ax.plot(metrics['lead_times'], metrics['csi'][thr], 
                    color=COLORS[i], marker=MARKERS[i], linestyle=LINE_STYLES[i%len(LINE_STYLES)], 
                    linewidth=3, markersize=12, label=f'{thr} mm/h')
            
            ax.axhline(0.5, color='purple', linestyle='--', linewidth=2, label='Threshold (0.5)')
            ax.set_ylim(0, 1.05)
            ax.legend(fontsize=FONT_SIZE)
            ax.set_title(f'Critical Success Index (CSI) - {thr} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        else:
            ax.text(0.5, 0.5, f'No valid CSI data for {thr} mm/h', 
                    ha='center', va='center', fontsize=FONT_SIZE+2)
        
        ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.set_ylabel('CSI', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xticks(SELECTED_LEADTIMES)
        ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'CSI_plot_{int(thr)}mm.png'), dpi=300, bbox_inches='tight')
        plt.close()

def plot_bias(metrics, output_dir, date_str):
    """Plot Bias for all thresholds as separate plots"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    # Create separate plots for each threshold
    for i, thr in enumerate(THRESHOLDS):
        fig, ax = plt.subplots(figsize=(12, 8))
        
        if any(not np.isnan(x) for x in metrics['bias'][thr]):
            ax.plot(metrics['lead_times'], metrics['bias'][thr], 
                    color=COLORS[i], marker=MARKERS[i], linestyle=LINE_STYLES[i%len(LINE_STYLES)], 
                    linewidth=3, markersize=12, label=f'{thr} mm/h')
            
            ax.axhline(1, color='k', linestyle='--', linewidth=3, label='Perfect bias')
            ax.legend(fontsize=FONT_SIZE)
            ax.set_title(f'Bias Score - {thr} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
            
            # Dynamically set y-limits
            bias_values = [x for x in metrics['bias'][thr] if not np.isnan(x)]
            if bias_values:
                max_bias = max(2, np.max(bias_values) * 1.1)
                min_bias = max(0, np.min(bias_values) * 0.9)
                ax.set_ylim(min_bias, max_bias)
        else:
            ax.text(0.5, 0.5, f'No valid Bias data for {thr} mm/h', 
                    ha='center', va='center', fontsize=FONT_SIZE+2)
        
        ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.set_ylabel('Bias Score', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.set_xticks(SELECTED_LEADTIMES)
        ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'Bias_plot_{int(thr)}mm.png'), dpi=300, bbox_inches='tight')
        plt.close()

def plot_data_availability(metrics, output_dir, date_str):
    """Plot data availability as bar chart with enhanced styling"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    # Create separate plots for each threshold
    for i, thr in enumerate(THRESHOLDS):
        fig, ax = plt.subplots(figsize=(14, 8))
        
        if any(x > 0 for x in metrics['data_availability'][thr]):
            x = np.arange(len(metrics['lead_times']))
            ax.bar(x, metrics['data_availability'][thr], 
                  width=0.6, color=COLORS[i], label=f'{thr} mm/h')
            
            ax.set_title(f'Data Availability - {thr} mm/h\n{formatted_date}', 
                        fontsize=FONT_SIZE+3, fontweight='bold')
            ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
            ax.set_ylabel('Number of Events', fontweight='bold', fontsize=FONT_SIZE+1)
            ax.legend(fontsize=FONT_SIZE)
            ax.set_xticks(x)
            ax.set_xticklabels(metrics['lead_times'], fontsize=FONT_SIZE)
            ax.grid(True, axis='y', linestyle='--', alpha=0.6)
            ax.tick_params(axis='y', labelsize=FONT_SIZE)
        else:
            ax.text(0.5, 0.5, f'No event data available for {thr} mm/h', 
                    ha='center', va='center', fontsize=FONT_SIZE+2)
        
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f'Data_Availability_{int(thr)}mm.png'), dpi=300, bbox_inches='tight')
        plt.close()

def save_results(metrics, output_dir, date_str):
    """Save all results with separate plots"""
    print("\nSaving results...")
    os.makedirs(output_dir, exist_ok=True)
    
    # Save metrics
    np.save(os.path.join(output_dir, "verification_metrics.npy"), metrics)
    
    # Create individual plots
    plot_rmse(metrics, output_dir, date_str)
    plot_correlation(metrics, output_dir, date_str)
    plot_fss(metrics, output_dir, date_str)
    plot_pod(metrics, output_dir, date_str)
    plot_far(metrics, output_dir, date_str)
    plot_csi(metrics, output_dir, date_str)
    plot_bias(metrics, output_dir, date_str)
    plot_data_availability(metrics, output_dir, date_str)
    
    # Save metrics summary table
    with open(os.path.join(output_dir, 'metrics_summary.txt'), 'w') as f:
        f.write("Verification Metrics Summary\n")
        f.write("="*50 + "\n")
        f.write(f"Date: {date_str}\n")
        f.write(f"Lead Times: {metrics['lead_times']}\n\n")
        
        # Write RMSE and Correlation header
        f.write("Basic Metrics\n")
        f.write("-"*30 + "\n")
        f.write("LeadTime  RMSE    Corr\n")
        f.write("----------------------\n")
        for i, lt in enumerate(metrics['lead_times']):
            rmse = metrics['rmse'][i] if i < len(metrics['rmse']) else np.nan
            corr = metrics['correlation'][i] if i < len(metrics['correlation']) else np.nan
            f.write(f"+{lt:3}min  {rmse:.3f}  {corr:.3f}\n")
        
        # Write threshold-based metrics
        for thr in THRESHOLDS:
            f.write(f"\nThreshold: {thr} mm/h\n")
            f.write("-"*30 + "\n")
            f.write("LeadTime  POD     FAR     CSI     Bias    FSS\n")
            f.write("--------------------------------------------\n")
            
            for i, lt in enumerate(metrics['lead_times']):
                pod = metrics['pod'][thr][i] if i < len(metrics['pod'][thr]) else np.nan
                far = metrics['far'][thr][i] if i < len(metrics['far'][thr]) else np.nan
                csi = metrics['csi'][thr][i] if i < len(metrics['csi'][thr]) else np.nan
                bias = metrics['bias'][thr][i] if i < len(metrics['bias'][thr]) else np.nan
                fss = metrics['fss'][thr][i] if i < len(metrics['fss'][thr]) else np.nan
                
                f.write(f"+{lt:3}min  {pod:.3f}  {far:.3f}  {csi:.3f}  {bias:.3f}  {fss:.3f}\n")
    
    print(f"Results saved to: {output_dir}")

def main():
    """Main workflow"""
    print(f"Starting verification for date: {DATE_STR}")
    print("="*50)
    
    try:
        # Load data
        print("\n[1/4] Loading nowcast input data...")
        R, _, metadata = load_santanu_data(NOWCAST_INPUT_DIR)
        
        # Perform nowcast
        print("\n[2/4] Performing nowcast...")
        R_f_steps = perform_nowcast(R, metadata)
        
        # Load verification data
        print("\n[3/4] Loading verification data...")
        R_o = load_verification_data(VERIFICATION_INPUT_DIR, N_LEADTIMES)
        R_o, _ = conversion.to_rainrate(R_o, metadata)
        
        # Calculate metrics
        print("\n[4/4] Calculating verification metrics...")
        metrics = calculate_metrics(R_f_steps, R_o, metadata)
        
        # Save results
        save_results(metrics, OUTPUT_DIR, DATE_STR)
        
        print("\nProcess completed successfully!")
        print(f"Results saved to: {OUTPUT_DIR}")
    
    except Exception as e:
        print(f"\nERROR: Process failed - {str(e)}")
        raise

if __name__ == "__main__":
    main()

Starting verification for date: 20221109

[1/4] Loading nowcast input data...

Loading data from: D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/INPUT_NOWCAST/
Successfully loaded 7 files

[2/4] Performing nowcast...

Starting nowcast process...
- Original data stats:
  Shape: (7, 734, 734)
  Min: -0.00, Max: 25.59
  NaN count: 0 (0.0%)
- Applying dB transform...
- Post-transform stats:
  Min: -0.00, Max: 25.59
  NaN count: 0 (0.0%)
- Calculating motion field...
- Running STEPS nowcast for 30 lead times...
Computing STEPS nowcast
-----------------------

Inputs
------
input dimensions: 734x734
km/pixel:         0.12
time step:        2 minutes

Methods
-------
extrapolation:          semilagrangian
bandpass filter:        gaussian
decomposition:          fft
noise generator:        nonparametric
noise adjustment:       no
velocity perturbator:   bps
conditional statistics: no
precip. mask method:    incremental
probability matching:   cdf
FFT method:             numpy
domain:           

In [4]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io
import glob
from datetime import datetime
import pandas as pd
from pysteps import io, nowcasts, verification
from pysteps.motion.lucaskanade import dense_lucaskanade
from pysteps.utils import conversion, dimension, transformation
from pysteps.postprocessing import ensemblestats
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.gridspec as gridspec
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configuration
DATE_STR = "20221109"  
NOWCAST_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/INPUT_NOWCAST/"
VERIFICATION_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/"
OUTPUT_DIR = "D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/verifications/"
N_LEADTIMES = 30  
N_ENS_MEMBERS = 10
SEED = 24
THRESHOLD = 20.0  # Single threshold for rainfall intensity (mm/h)
ACCUTIME = 2  # Time between frames in minutes
SELECTED_LEADTIMES = np.arange(10, 61, 10)  # From 10 to 60 minutes in 10-min steps

# Enhanced Visual Configuration matching the RMSE plot image
plt.style.use('default')  # Reset to default style
COLORS = {
    'rmse': '#1f77b4',      # Blue
    'correlation': '#ff7f0e', # Orange
    'fss': '#2ca02c',       # Green
    'pod': '#d62728',       # Red
    'far': '#9467bd',       # Purple
    'csi': '#8c564b',       # Brown
    'bias': '#e377c2',      # Pink
    'data': '#7f7f7f'       # Gray
}

MARKERS = {
    'rmse': 'o',
    'correlation': 's',
    'fss': 'D',
    'pod': '^',
    'far': 'v',
    'csi': '<',
    'bias': '>'
}

LINE_STYLES = {
    'rmse': '-',
    'correlation': '--',
    'fss': '-.',
    'pod': ':',
    'far': '-',
    'csi': '--',
    'bias': '-'
}

FONT_SIZE = 12
TITLE_FONT_SIZE = 14
LINE_WIDTH = 2.5
MARKER_SIZE = 8

plt.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.titlesize': TITLE_FONT_SIZE,
    'axes.labelsize': FONT_SIZE,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'figure.titlesize': TITLE_FONT_SIZE + 2,
    'figure.facecolor': 'white',
    'figure.edgecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.linewidth': 1.0,
    'grid.color': '0.9',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'xtick.color': 'black',
    'ytick.color': 'black',
    'xtick.major.size': 5,
    'xtick.major.width': 1,
    'ytick.major.size': 5,
    'ytick.major.width': 1,
    'lines.linewidth': LINE_WIDTH,
    'lines.markersize': MARKER_SIZE,
    'legend.framealpha': 0.8,
    'legend.edgecolor': '0.8'
})

def load_santanu_data(input_dir):
    """Load radar reflectivity data from .mat files"""
    print(f"\nLoading data from: {input_dir}")
    
    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Folder {input_dir} does not exist.")
    
    R = []
    times = []
    files_loaded = 0
    
    for filename in sorted(os.listdir(input_dir)):
        if filename.endswith('.mat'):
            try:
                file_path = os.path.join(input_dir, filename)
                mat_data = scipy.io.loadmat(file_path)
                
                if 'ZI' not in mat_data:
                    continue
                
                reflectivity = mat_data['ZI'] / 10000  # Normalize
                reflectivity[reflectivity == -1] = np.nan
                
                if np.all(np.isnan(reflectivity)):
                    print(f"Warning: All NaN values in {filename}")
                    continue
                
                R.append(reflectivity)
                files_loaded += 1
                
            except Exception as e:
                print(f"Error loading {filename}: {str(e)}")
                continue
    
    if not R:
        raise ValueError("No valid reflectivity data found.")
    
    print(f"Successfully loaded {files_loaded} files")
    
    metadata = {
        "institution": "Santanu",
        "accutime": ACCUTIME,
        "unit": "mm/h",
        "transform": "dB",
        "zerovalue": 0,
        "threshold": 0.1,
        'x1': 107, 'x2': 108,
        'y1': -7.2, 'y2': -6.4,
        'lat': np.linspace(-7.2, -6.4, R[0].shape[0]),
        'lon': np.linspace(107, 108, R[0].shape[1]),
        'projection': "+proj=merc +lon_0=0 +datum=WGS84 +units=m +no_defs",
        'yorigin': 'upper',
        "cartesian_map": True,
        'zr_a': 316.0, 'zr_b': 1.5,
    }
    
    return np.array(R), np.array(times), metadata

def perform_nowcast(R, metadata):
    """Perform STEPS nowcasting"""
    print("\nStarting nowcast process...")
    
    try:
        # Validate input data
        if np.all(np.isnan(R[-3:])):
            raise ValueError("Input data contains only NaN values")
            
        print("- Original data stats:")
        print(f"  Shape: {R.shape}")
        print(f"  Min: {np.nanmin(R):.2f}, Max: {np.nanmax(R):.2f}")
        print(f"  NaN count: {np.isnan(R).sum()} ({np.isnan(R).mean()*100:.1f}%)")

        # Transform data
        print("- Applying dB transform...")
        R, metadata = transformation.dB_transform(
            R, 
            metadata, 
            threshold=0.1, 
            zerovalue=-15.0
        )
        
        print("- Post-transform stats:")
        print(f"  Min: {np.nanmin(R):.2f}, Max: {np.nanmax(R):.2f}")
        print(f"  NaN count: {np.isnan(R).sum()} ({np.isnan(R).mean()*100:.1f}%)")
        
        if np.all(np.isnan(R[-3:])):
            raise ValueError("All NaN values after dB transform")

        # Replace remaining NaN values
        R[~np.isfinite(R)] = -15.0
        
        # Estimate motion field
        print("- Calculating motion field...")
        V = dense_lucaskanade(R[-3:])  # Only use last 3 frames for motion estimation
        
        # Validate motion field
        if np.all(np.isnan(V)):
            raise ValueError("Motion field calculation failed (all NaN values)")

        # Perform nowcast
        print(f"- Running STEPS nowcast for {N_LEADTIMES} lead times...")
        nowcast_method = nowcasts.get_method("steps")
        R_f_steps = nowcast_method(
            R[-3:, :, :],  # Use last 3 frames
            V,
            N_LEADTIMES,
            N_ENS_MEMBERS,
            n_cascade_levels=6,
            precip_thr=0.1,
            kmperpixel=0.12,
            timestep=metadata["accutime"],
            noise_method="nonparametric",
            vel_pert_method="bps",
            mask_method="incremental",
            seed=SEED,
        )
        
        # Back-transform to rain rates
        print("- Converting back to rain rates...")
        R_f_steps, _ = transformation.dB_transform(
            R_f_steps, 
            threshold=0.1,
            inverse=True
        )
        
        return R_f_steps
    
    except Exception as e:
        print(f"Error in nowcast process: {str(e)}")
        print("Debug info:")
        if 'R' in locals():
            print(f"R shape: {R.shape if hasattr(R, 'shape') else 'N/A'}")
            print(f"R valid values: {np.sum(np.isfinite(R)) if hasattr(R, 'shape') else 'N/A'}")
        if 'V' in locals():
            print(f"V shape: {V.shape if hasattr(V, 'shape') else 'N/A'}")
        raise

def load_verification_data(input_dir, n_leadtimes):
    """Load verification data"""
    print(f"\nLoading verification data from: {input_dir}")
    
    fns = glob.glob(os.path.join(input_dir, "*.mat"))
    fns.sort()
    
    if not fns:
        raise FileNotFoundError(f"No .mat files found in {input_dir}")
    
    if len(fns) < n_leadtimes:
        print(f"Warning: Only {len(fns)} observation files available (requested {n_leadtimes})")
    
    R_o = []
    valid_files = 0
    
    for fn in fns[-n_leadtimes:]:
        try:
            mat_data = scipy.io.loadmat(fn)
            if 'ZI' in mat_data:
                reflectivity = mat_data['ZI'] / 10000
                reflectivity[reflectivity == -1] = np.nan
                
                if np.any(~np.isnan(reflectivity)):
                    R_o.append(reflectivity)
                    valid_files += 1
                else:
                    print(f"Warning: All NaN values in {os.path.basename(fn)}")
        except Exception as e:
            print(f"Error loading {os.path.basename(fn)}: {str(e)}")
    
    print(f"Loaded {valid_files} valid verification files")
    return np.array(R_o)

def calculate_metrics(R_f_steps, R_o, metadata):
    """Calculate verification metrics"""
    print("\nCalculating verification metrics...")
    
    n_leadtimes = min(R_f_steps.shape[1], R_o.shape[0])
    lead_times = [(lt+1)*metadata['accutime'] for lt in range(n_leadtimes)]
    
    metrics = {
        'lead_times': lead_times,
        'rmse': [],
        'correlation': [],
        'fss': [],
        'bias': [],
        'pod': [],
        'far': [],
        'csi': [],
        'data_availability': [],
    }

    for lt in range(n_leadtimes):
        current_lead_time = lead_times[lt]
        print(f"\nProcessing lead time: +{current_lead_time} minutes")
        
        R_f_mean = np.mean(R_f_steps[:, lt, :, :], axis=0)
        R_o_lt = R_o[lt, :, :]
        
        # Basic metrics
        valid_mask = np.isfinite(R_f_mean) & np.isfinite(R_o_lt)
        valid_count = np.sum(valid_mask)
        
        if valid_count > 0:
            metrics['rmse'].append(sqrt(mean_squared_error(R_f_mean[valid_mask], R_o_lt[valid_mask])))
            metrics['correlation'].append(np.corrcoef(R_f_mean[valid_mask].flatten(), R_o_lt[valid_mask].flatten())[0, 1])
        else:
            print(f"Warning: No valid pixels for basic metrics at +{current_lead_time}min")
            metrics['rmse'].append(np.nan)
            metrics['correlation'].append(np.nan)
        
        # Threshold-based metrics
        obs_above = np.nansum(R_o_lt >= THRESHOLD)
        fcst_above = np.nansum(R_f_mean >= THRESHOLD)
        metrics['data_availability'].append(obs_above)
        
        print(f"  Threshold {THRESHOLD}mm/h - Obs above: {obs_above}, Fcst above: {fcst_above}")
        
        # FSS
        try:
            fss = verification.fss(R_f_mean, R_o_lt, THRESHOLD, scale=10)
            metrics['fss'].append(fss)
        except Exception as e:
            print(f"    FSS calculation failed: {str(e)}")
            metrics['fss'].append(np.nan)
        
        # Binary contingency metrics
        obs_binary = (R_o_lt >= THRESHOLD).astype(float)
        fcst_binary = (R_f_mean >= THRESHOLD).astype(float)
        
        valid = np.isfinite(obs_binary) & np.isfinite(fcst_binary)
        obs_binary = obs_binary[valid]
        fcst_binary = fcst_binary[valid]
        
        if len(obs_binary) > 0 and np.any(obs_binary > 0):
            hits = np.sum((fcst_binary == 1) & (obs_binary == 1))
            false_alarms = np.sum((fcst_binary == 1) & (obs_binary == 0))
            misses = np.sum((fcst_binary == 0) & (obs_binary == 1))
            
            denom = hits + misses
            metrics['pod'].append(hits / denom if denom > 0 else np.nan)
            
            denom = hits + false_alarms
            metrics['far'].append(false_alarms / denom if denom > 0 else np.nan)
            metrics['csi'].append(hits / (hits + false_alarms + misses) if (hits + false_alarms + misses) > 0 else np.nan)
            metrics['bias'].append((hits + false_alarms) / (hits + misses) if (hits + misses) > 0 else np.nan)
        else:
            print(f"    Insufficient data for contingency metrics at {THRESHOLD}mm/h")
            metrics['pod'].append(np.nan)
            metrics['far'].append(np.nan)
            metrics['csi'].append(np.nan)
            metrics['bias'].append(np.nan)
    
    return metrics

def create_landscape_plot():
    """Create a landscape-oriented figure with white background"""
    fig = plt.figure(figsize=(10, 6), facecolor='white')
    plt.subplots_adjust(left=0.12, right=0.95, top=0.88, bottom=0.12, hspace=0.3)
    return fig

def plot_rmse(metrics, output_dir, date_str):
    """Plot RMSE matching the provided image style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(not np.isnan(x) for x in metrics['rmse']):
        # Main plot line
        ax.plot(metrics['lead_times'], metrics['rmse'], 
                color=COLORS['rmse'], marker=MARKERS['rmse'], linestyle=LINE_STYLES['rmse'], 
                linewidth=LINE_WIDTH, markersize=MARKER_SIZE, label='RMSE',
                markerfacecolor='white', markeredgewidth=1.5, markeredgecolor=COLORS['rmse'])
        
        # Set limits and grid
        y_max = np.nanmax(metrics['rmse']) * 1.1
        ax.set_ylim(0, y_max if not np.isnan(y_max) else 15)
        ax.set_xlim(0, 60)
        
        # Grid and ticks
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
        ax.set_xticks(np.arange(0, 61, 10))
        
        # Highlight selected lead time (10 minutes as in the example)
        highlight_lt = 10
        if highlight_lt in metrics['lead_times']:
            idx = metrics['lead_times'].index(highlight_lt)
            if not np.isnan(metrics['rmse'][idx]):
                # Vertical line
                ax.axvline(x=highlight_lt, color='gray', linestyle=':', linewidth=1)
                # Point highlight
                ax.plot(highlight_lt, metrics['rmse'][idx], 
                        marker=MARKERS['rmse'], markersize=MARKER_SIZE+2,
                        markerfacecolor=COLORS['rmse'], markeredgewidth=1.5, markeredgecolor='black')
                # Annotation
                ax.annotate(f"Selected +{highlight_lt}min\nRMSE: {metrics['rmse'][idx]:.2f}",
                           xy=(highlight_lt, metrics['rmse'][idx]),
                           xytext=(15, 15), textcoords='offset points',
                           bbox=dict(boxstyle='round,pad=0.5', fc='white', alpha=0.9, edgecolor='0.8'),
                           arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))
        
        # Add legend
        ax.legend(loc='upper right', framealpha=0.9)
    else:
        ax.text(0.5, 0.5, 'No valid RMSE data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    # Labels and title
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('RMSE (mm/h)', fontweight='bold', color='black')
    ax.set_title(f'RMSE vs Lead Time - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'RMSE_plot.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_correlation(metrics, output_dir, date_str):
    """Plot Correlation with distinct style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(not np.isnan(x) for x in metrics['correlation']):
        ax.plot(metrics['lead_times'], metrics['correlation'], 
                color=COLORS['correlation'], marker=MARKERS['correlation'], linestyle=LINE_STYLES['correlation'], 
                linewidth=LINE_WIDTH, markersize=MARKER_SIZE, label='Correlation',
                markerfacecolor='white', markeredgewidth=1.5, markeredgecolor=COLORS['correlation'])
        
        ax.set_ylim(-0.1, 1.1)
        ax.legend(loc='lower right', framealpha=0.9)
        ax.axhline(0.5, color='gray', linestyle=':', linewidth=1.5, label='Reference')
        
        ax.set_xticks(np.arange(0, 61, 10))
        ax.set_xlim(0, 60)
        
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
    else:
        ax.text(0.5, 0.5, 'No valid Correlation data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('Correlation Coefficient', fontweight='bold', color='black')
    ax.set_title(f'Correlation vs Lead Time - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'Correlation_plot.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_fss(metrics, output_dir, date_str):
    """Plot FSS with distinct style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(not np.isnan(x) for x in metrics['fss']):
        ax.plot(metrics['lead_times'], metrics['fss'], 
                color=COLORS['fss'], marker=MARKERS['fss'], linestyle=LINE_STYLES['fss'], 
                linewidth=LINE_WIDTH, markersize=MARKER_SIZE, label='FSS',
                markerfacecolor='white', markeredgewidth=1.5, markeredgecolor=COLORS['fss'])
        
        ax.axhline(0.5, color='k', linestyle='--', linewidth=1.5, label='Skill threshold')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='lower left', framealpha=0.9)
        
        ax.set_xticks(np.arange(0, 61, 10))
        ax.set_xlim(0, 60)
        
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
    else:
        ax.text(0.5, 0.5, 'No valid FSS data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('FSS Score', fontweight='bold', color='black')
    ax.set_title(f'Fractions Skill Score (FSS) - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'FSS_plot.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_pod(metrics, output_dir, date_str):
    """Plot POD with distinct style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(not np.isnan(x) for x in metrics['pod']):
        ax.plot(metrics['lead_times'], metrics['pod'], 
                color=COLORS['pod'], marker=MARKERS['pod'], linestyle=LINE_STYLES['pod'], 
                linewidth=LINE_WIDTH, markersize=MARKER_SIZE, label='POD',
                markerfacecolor='white', markeredgewidth=1.5, markeredgecolor=COLORS['pod'])
        
        ax.axhline(0.6, color='r', linestyle='--', linewidth=1.5, label='Threshold (0.6)')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='lower left', framealpha=0.9)
        
        ax.set_xticks(np.arange(0, 61, 10))
        ax.set_xlim(0, 60)
        
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
    else:
        ax.text(0.5, 0.5, 'No valid POD data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('POD', fontweight='bold', color='black')
    ax.set_title(f'Probability of Detection (POD) - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'POD_plot.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_far(metrics, output_dir, date_str):
    """Plot FAR with distinct style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(not np.isnan(x) for x in metrics['far']):
        ax.plot(metrics['lead_times'], metrics['far'], 
                color=COLORS['far'], marker=MARKERS['far'], linestyle=LINE_STYLES['far'], 
                linewidth=LINE_WIDTH, markersize=MARKER_SIZE, label='FAR',
                markerfacecolor='white', markeredgewidth=1.5, markeredgecolor=COLORS['far'])
        
        ax.axhline(0.4, color='b', linestyle='--', linewidth=1.5, label='Threshold (0.4)')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='upper left', framealpha=0.9)
        
        ax.set_xticks(np.arange(0, 61, 10))
        ax.set_xlim(0, 60)
        
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
    else:
        ax.text(0.5, 0.5, 'No valid FAR data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('FAR', fontweight='bold', color='black')
    ax.set_title(f'False Alarm Ratio (FAR) - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'FAR_plot.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_csi(metrics, output_dir, date_str):
    """Plot CSI with distinct style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(not np.isnan(x) for x in metrics['csi']):
        ax.plot(metrics['lead_times'], metrics['csi'], 
                color=COLORS['csi'], marker=MARKERS['csi'], linestyle=LINE_STYLES['csi'], 
                linewidth=LINE_WIDTH, markersize=MARKER_SIZE, label='CSI',
                markerfacecolor='white', markeredgewidth=1.5, markeredgecolor=COLORS['csi'])
        
        ax.axhline(0.5, color='purple', linestyle='--', linewidth=1.5, label='Threshold (0.5)')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='lower left', framealpha=0.9)
        
        ax.set_xticks(np.arange(0, 61, 10))
        ax.set_xlim(0, 60)
        
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
    else:
        ax.text(0.5, 0.5, 'No valid CSI data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('CSI', fontweight='bold', color='black')
    ax.set_title(f'Critical Success Index (CSI) - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'CSI_plot.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_bias(metrics, output_dir, date_str):
    """Plot Bias with distinct style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(not np.isnan(x) for x in metrics['bias']):
        ax.plot(metrics['lead_times'], metrics['bias'], 
                color=COLORS['bias'], marker=MARKERS['bias'], linestyle=LINE_STYLES['bias'], 
                linewidth=LINE_WIDTH, markersize=MARKER_SIZE, label='Bias',
                markerfacecolor='white', markeredgewidth=1.5, markeredgecolor=COLORS['bias'])
        
        ax.axhline(1, color='k', linestyle='--', linewidth=1.5, label='Perfect bias')
        ax.legend(loc='upper left', framealpha=0.9)
        
        ax.set_xticks(np.arange(0, 61, 10))
        ax.set_xlim(0, 60)
        
        bias_values = [x for x in metrics['bias'] if not np.isnan(x)]
        
        if bias_values:
            max_bias = max(2, np.max(bias_values) * 1.1)
            min_bias = max(0, np.min(bias_values) * 0.9)
            ax.set_ylim(min_bias, max_bias)
        
        ax.grid(True, which='both', linestyle='--', alpha=0.5)
        ax.set_axisbelow(True)
    else:
        ax.text(0.5, 0.5, 'No valid Bias data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('Bias Score', fontweight='bold', color='black')
    ax.set_title(f'Bias Score - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'Bias_plot.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def plot_data_availability(metrics, output_dir, date_str):
    """Plot data availability with distinct style"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig = create_landscape_plot()
    ax = fig.add_subplot(111)
    ax.set_facecolor('white')
    
    if any(x > 0 for x in metrics['data_availability']):
        ax.bar(metrics['lead_times'], metrics['data_availability'], 
              color=COLORS['data'], edgecolor='white', linewidth=1, alpha=0.7)
        
        ax.set_xticks(np.arange(0, 61, 10))
        ax.set_xlim(0, 60)
        ax.grid(True, axis='y', linestyle='--', alpha=0.3)
        
        # Add value labels on top of bars
        for i, v in enumerate(metrics['data_availability']):
            if v > 0:
                ax.text(metrics['lead_times'][i], v + 0.5, str(v),
                        ha='center', va='bottom', fontsize=FONT_SIZE-1)
    else:
        ax.text(0.5, 0.5, 'No event data available', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Lead Time (minutes)', fontweight='bold', color='black')
    ax.set_ylabel('Number of Events', fontweight='bold', color='black')
    ax.set_title(f'Data Availability - {formatted_date}', fontweight='bold', pad=12, color='black')
    
    plt.savefig(os.path.join(output_dir, 'Data_Availability.png'), dpi=300, bbox_inches='tight', facecolor='white')
    plt.close()

def save_results(metrics, output_dir, date_str):
    """Save all results with distinct styles"""
    print("\nSaving results...")
    os.makedirs(output_dir, exist_ok=True)
    
    np.save(os.path.join(output_dir, "verification_metrics.npy"), metrics)
    
    plot_rmse(metrics, output_dir, date_str)
    plot_correlation(metrics, output_dir, date_str)
    plot_fss(metrics, output_dir, date_str)
    plot_pod(metrics, output_dir, date_str)
    plot_far(metrics, output_dir, date_str)
    plot_csi(metrics, output_dir, date_str)
    plot_bias(metrics, output_dir, date_str)
    plot_data_availability(metrics, output_dir, date_str)
    
    with open(os.path.join(output_dir, 'metrics_summary.txt'), 'w') as f:
        f.write("Verification Metrics Summary\n")
        f.write("="*50 + "\n")
        f.write(f"Date: {date_str}\n")
        f.write(f"Threshold: {THRESHOLD} mm/h\n")
        f.write(f"Lead Times: {metrics['lead_times']}\n\n")
        
        f.write("Basic Metrics\n")
        f.write("-"*30 + "\n")
        f.write("LeadTime  RMSE    Corr\n")
        f.write("----------------------\n")
        for i, lt in enumerate(metrics['lead_times']):
            rmse = metrics['rmse'][i] if i < len(metrics['rmse']) else np.nan
            corr = metrics['correlation'][i] if i < len(metrics['correlation']) else np.nan
            f.write(f"+{lt:3}min  {rmse:.3f}  {corr:.3f}\n")
        
        f.write(f"\nThreshold: {THRESHOLD} mm/h\n")
        f.write("-"*30 + "\n")
        f.write("LeadTime  POD     FAR     CSI     Bias    FSS\n")
        f.write("--------------------------------------------\n")
        
        for i, lt in enumerate(metrics['lead_times']):
            pod = metrics['pod'][i] if i < len(metrics['pod']) else np.nan
            far = metrics['far'][i] if i < len(metrics['far']) else np.nan
            csi = metrics['csi'][i] if i < len(metrics['csi']) else np.nan
            bias = metrics['bias'][i] if i < len(metrics['bias']) else np.nan
            fss = metrics['fss'][i] if i < len(metrics['fss']) else np.nan
            
            f.write(f"+{lt:3}min  {pod:.3f}  {far:.3f}  {csi:.3f}  {bias:.3f}  {fss:.3f}\n")
    
    print(f"Results saved to: {output_dir}")

def main():
    """Main workflow"""
    print(f"Starting verification for date: {DATE_STR}")
    print("="*50)
    
    try:
        # Load data
        print("\n[1/4] Loading nowcast input data...")
        R, _, metadata = load_santanu_data(NOWCAST_INPUT_DIR)
        
        # Perform nowcast
        print("\n[2/4] Performing nowcast...")
        R_f_steps = perform_nowcast(R, metadata)
        
        # Load verification data
        print("\n[3/4] Loading verification data...")
        R_o = load_verification_data(VERIFICATION_INPUT_DIR, N_LEADTIMES)
        R_o, _ = conversion.to_rainrate(R_o, metadata)
        
        # Calculate metrics
        print("\n[4/4] Calculating verification metrics...")
        metrics = calculate_metrics(R_f_steps, R_o, metadata)
        
        # Save results
        save_results(metrics, OUTPUT_DIR, DATE_STR)
        
        print("\nProcess completed successfully!")
        print(f"Results saved to: {OUTPUT_DIR}")
    
    except Exception as e:
        print(f"\nERROR: Process failed - {str(e)}")
        raise

if __name__ == "__main__":
    main()

Starting verification for date: 20221109

[1/4] Loading nowcast input data...

Loading data from: D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/INPUT_NOWCAST/
Successfully loaded 7 files

[2/4] Performing nowcast...

Starting nowcast process...
- Original data stats:
  Shape: (7, 734, 734)
  Min: -0.00, Max: 25.59
  NaN count: 0 (0.0%)
- Applying dB transform...
- Post-transform stats:
  Min: -0.00, Max: 25.59
  NaN count: 0 (0.0%)
- Calculating motion field...
- Running STEPS nowcast for 30 lead times...
Computing STEPS nowcast
-----------------------

Inputs
------
input dimensions: 734x734
km/pixel:         0.12
time step:        2 minutes

Methods
-------
extrapolation:          semilagrangian
bandpass filter:        gaussian
decomposition:          fft
noise generator:        nonparametric
noise adjustment:       no
velocity perturbator:   bps
conditional statistics: no
precip. mask method:    incremental
probability matching:   cdf
FFT method:             numpy
domain:           

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy.io
import glob
from datetime import datetime
import pandas as pd
from pysteps import io, nowcasts, verification
from pysteps.motion.lucaskanade import dense_lucaskanade
from pysteps.utils import conversion, dimension, transformation
from pysteps.postprocessing import ensemblestats
from sklearn.metrics import mean_squared_error
from math import sqrt
import matplotlib.gridspec as gridspec
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Configuration
DATE_STR = "20221109"  
NOWCAST_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/INPUT_NOWCAST/"
VERIFICATION_INPUT_DIR = "D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/"
OUTPUT_DIR = "D:/College/MENEA/FIXED_DATA/OUTPUT/20221109/verifications/"
N_LEADTIMES = 30  
N_ENS_MEMBERS = 10
SEED = 24
THRESHOLD = 20.0  # Single rainfall intensity threshold (mm/h)
ACCUTIME = 2  # Time between frames in minutes
SELECTED_LEADTIMES = [10, 20, 30, 40, 50, 60]  

# Enhanced Style configuration
plt.style.use('seaborn-v0_8-darkgrid')
COLOR = 'blue'  # Warna utama untuk plot threshold tunggal
MARKER = 'o'
LINE_STYLE = '-'
FONT_SIZE = 28  # Increased base font size
plt.rcParams.update({
    'font.size': FONT_SIZE,
    'axes.titlesize': FONT_SIZE + 2,
    'axes.labelsize': FONT_SIZE + 1,
    'xtick.labelsize': FONT_SIZE,
    'ytick.labelsize': FONT_SIZE,
    'legend.fontsize': FONT_SIZE,
    'figure.titlesize': FONT_SIZE + 4,
    'figure.facecolor': 'white',
    'figure.edgecolor': 'white'
})

def set_xaxis_ticks(ax, lead_times):
    """Set x-axis ticks at 10-minute intervals"""
    max_lt = max(lead_times) if lead_times else 60
    xticks = np.arange(0, max_lt + 1, 10)
    ax.set_xticks(xticks)
    ax.set_xticklabels([f"{int(x)}" for x in xticks])
    ax.set_xlim(0, max_lt)

def load_santanu_data(input_dir):
    """Load radar reflectivity data from .mat files"""
    print(f"\nLoading data from: {input_dir}")
    
    if not os.path.exists(input_dir):
        raise FileNotFoundError(f"Folder {input_dir} does not exist.")
    
    R = []
    times = []
    files_loaded = 0
    
    for filename in sorted(os.listdir(input_dir)):
        if filename.endswith('.mat'):
            try:
                file_path = os.path.join(input_dir, filename)
                mat_data = scipy.io.loadmat(file_path)
                
                if 'ZI' not in mat_data:
                    continue
                
                reflectivity = mat_data['ZI'] / 10000  # Normalize
                reflectivity[reflectivity == -1] = np.nan
                
                if np.all(np.isnan(reflectivity)):
                    print(f"Warning: All NaN values in {filename}")
                    continue
                
                R.append(reflectivity)
                files_loaded += 1
                
            except Exception as e:
                print(f"Error loading {filename}: {str(e)}")
                continue
    
    if not R:
        raise ValueError("No valid reflectivity data found.")
    
    print(f"Successfully loaded {files_loaded} files")
    
    metadata = {
        "institution": "Santanu",
        "accutime": ACCUTIME,
        "unit": "mm/h",
        "transform": "dB",
        "zerovalue": 0,
        "threshold": 0.1,
        'x1': 107, 'x2': 108,
        'y1': -7.2, 'y2': -6.4,
        'lat': np.linspace(-7.2, -6.4, R[0].shape[0]),
        'lon': np.linspace(107, 108, R[0].shape[1]),
        'projection': "+proj=merc +lon_0=0 +datum=WGS84 +units=m +no_defs",
        'yorigin': 'upper',
        "cartesian_map": True,
        'zr_a': 316.0, 'zr_b': 1.5,
    }
    
    return np.array(R), np.array(times), metadata

def perform_nowcast(R, metadata):
    """Perform STEPS nowcasting"""
    print("\nStarting nowcast process...")
    
    try:
        # Validate input data
        if np.all(np.isnan(R[-3:])):
            raise ValueError("Input data contains only NaN values")
            
        print("- Original data stats:")
        print(f"  Shape: {R.shape}")
        print(f"  Min: {np.nanmin(R):.2f}, Max: {np.nanmax(R):.2f}")
        print(f"  NaN count: {np.isnan(R).sum()} ({np.isnan(R).mean()*100:.1f}%)")

        # Transform data
        print("- Applying dB transform...")
        R, metadata = transformation.dB_transform(
            R, 
            metadata, 
            threshold=0.1, 
            zerovalue=-15.0
        )
        
        print("- Post-transform stats:")
        print(f"  Min: {np.nanmin(R):.2f}, Max: {np.nanmax(R):.2f}")
        print(f"  NaN count: {np.isnan(R).sum()} ({np.isnan(R).mean()*100:.1f}%)")
        
        if np.all(np.isnan(R[-3:])):
            raise ValueError("All NaN values after dB transform")

        # Replace remaining NaN values
        R[~np.isfinite(R)] = -15.0
        
        # Estimate motion field
        print("- Calculating motion field...")
        V = dense_lucaskanade(R[-3:])  # Only use last 3 frames for motion estimation
        
        # Validate motion field
        if np.all(np.isnan(V)):
            raise ValueError("Motion field calculation failed (all NaN values)")

        # Perform nowcast
        print(f"- Running STEPS nowcast for {N_LEADTIMES} lead times...")
        nowcast_method = nowcasts.get_method("steps")
        R_f_steps = nowcast_method(
            R[-3:, :, :],  # Use last 3 frames
            V,
            N_LEADTIMES,
            N_ENS_MEMBERS,
            n_cascade_levels=6,
            precip_thr=0.1,
            kmperpixel=0.12,
            timestep=metadata["accutime"],
            noise_method="nonparametric",
            vel_pert_method="bps",
            mask_method="incremental",
            seed=SEED,
        )
        
        # Back-transform to rain rates
        print("- Converting back to rain rates...")
        R_f_steps, _ = transformation.dB_transform(
            R_f_steps, 
            threshold=0.1,
            inverse=True
        )
        
        return R_f_steps
    
    except Exception as e:
        print(f"Error in nowcast process: {str(e)}")
        print("Debug info:")
        if 'R' in locals():
            print(f"R shape: {R.shape if hasattr(R, 'shape') else 'N/A'}")
            print(f"R valid values: {np.sum(np.isfinite(R)) if hasattr(R, 'shape') else 'N/A'}")
        if 'V' in locals():
            print(f"V shape: {V.shape if hasattr(V, 'shape') else 'N/A'}")
        raise

def load_verification_data(input_dir, n_leadtimes):
    """Load verification data"""
    print(f"\nLoading verification data from: {input_dir}")
    
    fns = glob.glob(os.path.join(input_dir, "*.mat"))
    fns.sort()
    
    if not fns:
        raise FileNotFoundError(f"No .mat files found in {input_dir}")
    
    if len(fns) < n_leadtimes:
        print(f"Warning: Only {len(fns)} observation files available (requested {n_leadtimes})")
    
    R_o = []
    valid_files = 0
    
    for fn in fns[-n_leadtimes:]:
        try:
            mat_data = scipy.io.loadmat(fn)
            if 'ZI' in mat_data:
                reflectivity = mat_data['ZI'] / 10000
                reflectivity[reflectivity == -1] = np.nan
                
                if np.any(~np.isnan(reflectivity)):
                    R_o.append(reflectivity)
                    valid_files += 1
                else:
                    print(f"Warning: All NaN values in {os.path.basename(fn)}")
        except Exception as e:
            print(f"Error loading {os.path.basename(fn)}: {str(e)}")
    
    print(f"Loaded {valid_files} valid verification files")
    return np.array(R_o)

def calculate_metrics(R_f_steps, R_o, metadata):
    """Calculate verification metrics"""
    print("\nCalculating verification metrics...")
    
    n_leadtimes = min(R_f_steps.shape[1], R_o.shape[0])
    lead_times = [(lt+1)*metadata['accutime'] for lt in range(n_leadtimes)]
    
    metrics = {
        'lead_times': lead_times,
        'rmse': [],
        'correlation': [],
        'fss': [],
        'bias': [],
        'pod': [],
        'far': [],
        'csi': [],
        'data_availability': [],
    }

    for lt in range(n_leadtimes):
        current_lead_time = lead_times[lt]
        print(f"\nProcessing lead time: +{current_lead_time} minutes")
        
        R_f_mean = np.mean(R_f_steps[:, lt, :, :], axis=0)
        R_o_lt = R_o[lt, :, :]
        
        # Basic metrics
        valid_mask = np.isfinite(R_f_mean) & np.isfinite(R_o_lt)
        valid_count = np.sum(valid_mask)
        
        if valid_count > 0:
            metrics['rmse'].append(sqrt(mean_squared_error(R_f_mean[valid_mask], R_o_lt[valid_mask])))
            metrics['correlation'].append(np.corrcoef(R_f_mean[valid_mask].flatten(), R_o_lt[valid_mask].flatten())[0, 1])
        else:
            print(f"Warning: No valid pixels for basic metrics at +{current_lead_time}min")
            metrics['rmse'].append(np.nan)
            metrics['correlation'].append(np.nan)
        
        # Threshold-based metrics
        thr = THRESHOLD
        obs_above = np.nansum(R_o_lt >= thr)
        fcst_above = np.nansum(R_f_mean >= thr)
        metrics['data_availability'].append(obs_above)
        
        print(f"  Threshold {thr}mm/h - Obs above: {obs_above}, Fcst above: {fcst_above}")
        
        # FSS
        try:
            fss = verification.fss(R_f_mean, R_o_lt, thr, scale=10)
            metrics['fss'].append(fss)
        except Exception as e:
            print(f"    FSS calculation failed: {str(e)}")
            metrics['fss'].append(np.nan)
        
        # Binary contingency metrics
        obs_binary = (R_o_lt >= thr).astype(float)
        fcst_binary = (R_f_mean >= thr).astype(float)
        
        valid = np.isfinite(obs_binary) & np.isfinite(fcst_binary)
        obs_binary = obs_binary[valid]
        fcst_binary = fcst_binary[valid]
        
        if len(obs_binary) > 0 and np.any(obs_binary > 0):
            hits = np.sum((fcst_binary == 1) & (obs_binary == 1))
            false_alarms = np.sum((fcst_binary == 1) & (obs_binary == 0))
            misses = np.sum((fcst_binary == 0) & (obs_binary == 1))
            
            denom = hits + misses
            metrics['pod'].append(hits / denom if denom > 0 else np.nan)
            
            denom = hits + false_alarms
            metrics['far'].append(false_alarms / denom if denom > 0 else np.nan)
            metrics['csi'].append(hits / (hits + false_alarms + misses) if (hits + false_alarms + misses) > 0 else np.nan)
            metrics['bias'].append((hits + false_alarms) / (hits + misses) if (hits + misses) > 0 else np.nan)
        else:
            print(f"    Insufficient data for contingency metrics at {thr}mm/h")
            metrics['pod'].append(np.nan)
            metrics['far'].append(np.nan)
            metrics['csi'].append(np.nan)
            metrics['bias'].append(np.nan)
    
    return metrics

def plot_rmse(metrics, output_dir, date_str):
    """Plot RMSE with enhanced styling and larger text"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    if any(not np.isnan(x) for x in metrics['rmse']):
        ax.plot(metrics['lead_times'], metrics['rmse'], 
                color='purple', marker='o', linestyle='-',  
                linewidth=3, markersize=12, label='RMSE')  
        
        # Highlight selected lead times
        for lt in SELECTED_LEADTIMES:
            if lt in metrics['lead_times']:
                idx = metrics['lead_times'].index(lt)
                if not np.isnan(metrics['rmse'][idx]):
                    ax.plot(lt, metrics['rmse'][idx], 'ro', markersize=14, 
                            markeredgecolor='k', linewidth=2,
                            label=f'Selected +{lt}min' if lt == SELECTED_LEADTIMES[0] else "")
        
        ax.set_ylim(0, np.nanmax(metrics['rmse']) * 1.1)
        ax.legend(fontsize=FONT_SIZE)
        
        # Set x-axis ticks at 10-minute intervals
        set_xaxis_ticks(ax, metrics['lead_times'])
    else:
        ax.text(0.5, 0.5, 'No valid RMSE data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('RMSE (mm/h)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_title(f'RMSE vs Lead Time\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.tick_params(axis='both', which='major', labelsize=32)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'RMSE_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_correlation(metrics, output_dir, date_str):
    """Plot Correlation with enhanced styling"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    if any(not np.isnan(x) for x in metrics['correlation']):
        ax.plot(metrics['lead_times'], metrics['correlation'], 
                color='#ff7f0e', marker='s', linestyle='--', 
                linewidth=3, markersize=12, label='Correlation')
        
        # Highlight selected lead times
        for lt in SELECTED_LEADTIMES:
            if lt in metrics['lead_times']:
                idx = metrics['lead_times'].index(lt)
                if not np.isnan(metrics['correlation'][idx]):
                    ax.plot(lt, metrics['correlation'][idx], 'ro', markersize=14, 
                            markeredgecolor='k', linewidth=2, label=f'Selected +{lt}min' if lt == SELECTED_LEADTIMES[0] else "")
        
        ax.set_ylim(-0.1, 1.1)
        ax.legend(fontsize=FONT_SIZE)
        ax.axhline(0.5, color='gray', linestyle=':', linewidth=2, label='Reference')
        
        # Set x-axis ticks at 10-minute intervals
        set_xaxis_ticks(ax, metrics['lead_times'])
    else:
        ax.text(0.5, 0.5, 'No valid Correlation data', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('Correlation Coefficient', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_title(f'Correlation vs Lead Time\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.tick_params(axis='both', which='major', labelsize=32)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'Correlation_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_fss(metrics, output_dir, date_str):
    """Plot FSS for single threshold"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    if any(not np.isnan(x) for x in metrics['fss']):
        ax.plot(metrics['lead_times'], metrics['fss'], 
                color=COLOR, marker=MARKER, linestyle=LINE_STYLE, 
                linewidth=3, markersize=12)
        
        ax.axhline(0.5, color='k', linestyle='--', linewidth=3, label='Skill threshold')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=FONT_SIZE)
        ax.set_title(f'Fractions Skill Score (FSS) - {THRESHOLD} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        
        # Set x-axis ticks at 10-minute intervals
        set_xaxis_ticks(ax, metrics['lead_times'])
    else:
        ax.text(0.5, 0.5, f'No valid FSS data for {THRESHOLD} mm/h', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('FSS Score', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.tick_params(axis='both', which='major', labelsize=32)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'FSS_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_pod(metrics, output_dir, date_str):
    """Plot Probability of Detection (POD)"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    if any(not np.isnan(x) for x in metrics['pod']):
        ax.plot(metrics['lead_times'], metrics['pod'], 
                color='blue', marker=MARKER, linestyle=LINE_STYLE,
                linewidth=3, markersize=12)
        
        ax.axhline(0.6, color='r', linestyle='--', linewidth=2, label='Threshold (0.6)')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=FONT_SIZE)
        ax.set_title(f'Probability of Detection (POD) - {THRESHOLD} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        
        # Set x-axis ticks at 10-minute intervals
        set_xaxis_ticks(ax, metrics['lead_times'])
    else:
        ax.text(0.5, 0.5, f'No valid POD data for {THRESHOLD} mm/h', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('POD', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.tick_params(axis='both', which='major', labelsize=32)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'POD_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_far(metrics, output_dir, date_str):
    """Plot False Alarm Ratio (FAR)"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    if any(not np.isnan(x) for x in metrics['far']):
        ax.plot(metrics['lead_times'], metrics['far'], 
                color='red', marker=MARKER, linestyle=LINE_STYLE,
                linewidth=3, markersize=12)
        
        ax.axhline(0.4, color='b', linestyle='--', linewidth=2, label='Threshold (0.4)')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=FONT_SIZE)
        ax.set_title(f'False Alarm Ratio (FAR) - {THRESHOLD} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        
        # Set x-axis ticks at 10-minute intervals
        set_xaxis_ticks(ax, metrics['lead_times'])
    else:
        ax.text(0.5, 0.5, f'No valid FAR data for {THRESHOLD} mm/h', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('FAR', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.tick_params(axis='both', which='major', labelsize=32)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'FAR_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_csi(metrics, output_dir, date_str):
    """Plot Critical Success Index (CSI)"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    if any(not np.isnan(x) for x in metrics['csi']):
        ax.plot(metrics['lead_times'], metrics['csi'], 
                color='green', marker=MARKER, linestyle=LINE_STYLE,  
                linewidth=3, markersize=12)
        
        ax.axhline(0.5, color='purple', linestyle='--', linewidth=2, label='Threshold (0.5)')
        ax.set_ylim(0, 1.05)
        ax.legend(fontsize=FONT_SIZE)
        ax.set_title(f'Critical Success Index (CSI) - {THRESHOLD} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        
        # Set x-axis ticks at 10-minute intervals
        set_xaxis_ticks(ax, metrics['lead_times'])
    else:
        ax.text(0.5, 0.5, f'No valid CSI data for {THRESHOLD} mm/h', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('CSI', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.tick_params(axis='both', which='major', labelsize=32)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'CSI_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_bias(metrics, output_dir, date_str):
    """Plot Bias"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(12, 8))
    
    if any(not np.isnan(x) for x in metrics['bias']):
        ax.plot(metrics['lead_times'], metrics['bias'], 
                color=COLOR, marker=MARKER, linestyle=LINE_STYLE, 
                linewidth=3, markersize=12)
        
        ax.axhline(1, color='k', linestyle='--', linewidth=3, label='Perfect bias')
        ax.legend(fontsize=FONT_SIZE)
        ax.set_title(f'Bias Score - {THRESHOLD} mm/h\n{formatted_date}', fontsize=FONT_SIZE+3, fontweight='bold')
        
        # Dynamically set y-limits
        bias_values = [x for x in metrics['bias'] if not np.isnan(x)]
        if bias_values:
            max_bias = max(2, np.max(bias_values) * 1.1)
            min_bias = max(0, np.min(bias_values) * 0.9)
            ax.set_ylim(min_bias, max_bias)
            
        # Set x-axis ticks at 10-minute intervals
        set_xaxis_ticks(ax, metrics['lead_times'])
    else:
        ax.text(0.5, 0.5, f'No valid Bias data for {THRESHOLD} mm/h', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.set_ylabel('Bias Score', fontweight='bold', fontsize=FONT_SIZE+1)
    ax.grid(True, linestyle='--', alpha=0.6)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'Bias_plot.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_data_availability(metrics, output_dir, date_str):
    """Plot data availability as bar chart with enhanced styling"""
    date_obj = datetime.strptime(date_str, "%Y%m%d")
    formatted_date = date_obj.strftime("%d %B %Y")
    
    fig, ax = plt.subplots(figsize=(16, 8))
    
    if any(x > 0 for x in metrics['data_availability']):
        x = np.arange(len(metrics['lead_times']))
        ax.bar(x, metrics['data_availability'], 
              width=0.6, color=COLOR, label=f'{THRESHOLD} mm/h')
        
        # Set x-axis ticks at 10-minute intervals
        xticks = []
        xticklabels = []
        for i, lt in enumerate(metrics['lead_times']):
            if lt % 10 == 0:  # Only show every 10 minutes
                xticks.append(i)
                xticklabels.append(str(lt))
        
        ax.set_xticks(xticks)
        ax.set_xticklabels(xticklabels, fontsize=FONT_SIZE)
        
        ax.set_title(f'Data Availability - {THRESHOLD} mm/h\n{formatted_date}', 
                    fontsize=FONT_SIZE+3, fontweight='bold')
        ax.set_xlabel('Experimental Time Lead (minutes)', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.set_ylabel('Number of Events', fontweight='bold', fontsize=FONT_SIZE+1)
        ax.legend(fontsize=FONT_SIZE)
        ax.grid(True, axis='y', linestyle='--', alpha=0.6)
        ax.tick_params(axis='y', labelsize=FONT_SIZE)
    else:
        ax.text(0.5, 0.5, f'No event data available for {THRESHOLD} mm/h', 
                ha='center', va='center', fontsize=FONT_SIZE+2)
    
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'Data_Availability.png'), dpi=300, bbox_inches='tight')
    plt.close()

def save_results(metrics, output_dir, date_str):
    """Save all results with separate plots"""
    print("\nSaving results...")
    os.makedirs(output_dir, exist_ok=True)
    
    # Save metrics
    np.save(os.path.join(output_dir, "verification_metrics.npy"), metrics)
    
    # Create individual plots
    plot_rmse(metrics, output_dir, date_str)
    plot_correlation(metrics, output_dir, date_str)
    plot_fss(metrics, output_dir, date_str)
    plot_pod(metrics, output_dir, date_str)
    plot_far(metrics, output_dir, date_str)
    plot_csi(metrics, output_dir, date_str)
    plot_bias(metrics, output_dir, date_str)
    plot_data_availability(metrics, output_dir, date_str)
    
    # Save metrics summary table
    with open(os.path.join(output_dir, 'metrics_summary.txt'), 'w') as f:
        f.write("Verification Metrics Summary\n")
        f.write("="*50 + "\n")
        f.write(f"Date: {date_str}\n")
        f.write(f"Lead Times: {metrics['lead_times']}\n\n")
        f.write(f"Threshold: {THRESHOLD} mm/h\n")
        
        # Write RMSE and Correlation header
        f.write("Basic Metrics\n")
        f.write("-"*30 + "\n")
        f.write("LeadTime  RMSE    Corr\n")
        f.write("----------------------\n")
        for i, lt in enumerate(metrics['lead_times']):
            rmse = metrics['rmse'][i] if i < len(metrics['rmse']) else np.nan
            corr = metrics['correlation'][i] if i < len(metrics['correlation']) else np.nan
            f.write(f"+{lt:3}min  {rmse:.3f}  {corr:.3f}\n")
        
        # Write threshold-based metrics
        f.write("\nThreshold Metrics\n")
        f.write("-"*30 + "\n")
        f.write("LeadTime  POD     FAR     CSI     Bias    FSS\n")
        f.write("--------------------------------------------\n")
        
        for i, lt in enumerate(metrics['lead_times']):
            pod = metrics['pod'][i] if i < len(metrics['pod']) else np.nan
            far = metrics['far'][i] if i < len(metrics['far']) else np.nan
            csi = metrics['csi'][i] if i < len(metrics['csi']) else np.nan
            bias = metrics['bias'][i] if i < len(metrics['bias']) else np.nan
            fss = metrics['fss'][i] if i < len(metrics['fss']) else np.nan
            
            f.write(f"+{lt:3}min  {pod:.3f}  {far:.3f}  {csi:.3f}  {bias:.3f}  {fss:.3f}\n")
    
    print(f"Results saved to: {output_dir}")

def main():
    """Main workflow"""
    print(f"Starting verification for date: {DATE_STR}")
    print("="*50)
    
    try:
        # Load data
        print("\n[1/4] Loading nowcast input data...")
        R, _, metadata = load_santanu_data(NOWCAST_INPUT_DIR)
        
        # Perform nowcast
        print("\n[2/4] Performing nowcast...")
        R_f_steps = perform_nowcast(R, metadata)
        
        # Load verification data
        print("\n[3/4] Loading verification data...")
        R_o = load_verification_data(VERIFICATION_INPUT_DIR, N_LEADTIMES)
        R_o, _ = conversion.to_rainrate(R_o, metadata)
        
        # Calculate metrics
        print("\n[4/4] Calculating verification metrics...")
        metrics = calculate_metrics(R_f_steps, R_o, metadata)
        
        # Save results
        save_results(metrics, OUTPUT_DIR, DATE_STR)
        
        print("\nProcess completed successfully!")
        print(f"Results saved to: {OUTPUT_DIR}")
    
    except Exception as e:
        print(f"\nERROR: Process failed - {str(e)}")
        raise

if __name__ == "__main__":
    main()

Starting verification for date: 20221109

[1/4] Loading nowcast input data...

Loading data from: D:/College/MENEA/FIXED_DATA/RUN_TEST/20221109/INPUT_NOWCAST/
Successfully loaded 7 files

[2/4] Performing nowcast...

Starting nowcast process...
- Original data stats:
  Shape: (7, 734, 734)
  Min: -0.00, Max: 25.59
  NaN count: 0 (0.0%)
- Applying dB transform...
- Post-transform stats:
  Min: -0.00, Max: 25.59
  NaN count: 0 (0.0%)
- Calculating motion field...
- Running STEPS nowcast for 30 lead times...
Computing STEPS nowcast
-----------------------

Inputs
------
input dimensions: 734x734
km/pixel:         0.12
time step:        2 minutes

Methods
-------
extrapolation:          semilagrangian
bandpass filter:        gaussian
decomposition:          fft
noise generator:        nonparametric
noise adjustment:       no
velocity perturbator:   bps
conditional statistics: no
precip. mask method:    incremental
probability matching:   cdf
FFT method:             numpy
domain:           